# Python Interview Prep Guide — Runnable Notebook

Use this prep guide like a real interview notebook: read the explanation, run the code immediately underneath it, inspect the output, then change one thing and rerun.

The structure is intentionally Jupyter-style: every concept explanation is followed by the code cell that proves it. The examples cover parsing, aggregation, joins, metrics, visualisation, telemetry, benchmarking, output narration, and debugging.


## 0. Setup — imports and tiny interview datasets

First create small local fixtures. In a real interview, this is where you would load the files the interviewer provides. Keeping the data tiny makes every transformation easy to inspect.


In [ ]:
from pathlib import Path
import csv
import json
import math
import statistics as stats
from collections import Counter, defaultdict
from pprint import pprint

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

DATA_DIR = Path("python_prep_data")
DATA_DIR.mkdir(exist_ok=True)

sample_results = [
    {"model": "alpha", "task": "classify", "score": 0.91, "passed": True, "latency_ms": 820, "cost_usd": 0.012},
    {"model": "alpha", "task": "summarize", "score": 0.84, "passed": True, "latency_ms": 1290, "cost_usd": 0.018},
    {"model": "bravo", "task": "classify", "score": 0.89, "passed": True, "latency_ms": 610, "cost_usd": 0.009},
    {"model": "bravo", "task": "summarize", "score": 0.78, "passed": False, "latency_ms": 980, "cost_usd": 0.011},
    {"model": "charlie", "task": "classify", "score": 0.93, "passed": True, "latency_ms": 1500, "cost_usd": 0.021},
]

telemetry_rows = [
    {"request_id": "r1", "provider": "openai", "model": "alpha", "status_code": 200, "latency_ms": 820, "cost_usd": 0.012, "span_id": "s1"},
    {"request_id": "r2", "provider": "openai", "model": "alpha", "status_code": 200, "latency_ms": 1290, "cost_usd": 0.018, "span_id": "s2"},
    {"request_id": "r3", "provider": "anthropic", "model": "bravo", "status_code": 500, "latency_ms": 980, "cost_usd": 0.011, "span_id": "s3"},
    {"request_id": "r4", "provider": "anthropic", "model": "bravo", "status_code": 200, "latency_ms": 610, "cost_usd": 0.009, "span_id": ""},
    {"request_id": "r5", "provider": "local", "model": "charlie", "status_code": 200, "latency_ms": 1500, "cost_usd": 0.000, "span_id": "s5"},
]

(DATA_DIR / "sample_results.json").write_text(json.dumps(sample_results, indent=2))
with (DATA_DIR / "telemetry.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(telemetry_rows[0]))
    writer.writeheader()
    writer.writerows(telemetry_rows)
with (DATA_DIR / "batch_outputs.jsonl").open("w") as f:
    for row in sample_results[:3]:
        f.write(json.dumps(row) + "\n")
    f.write("{bad json line}\n")
    f.write(json.dumps(sample_results[3]) + "\n")

print("Created fixtures in", DATA_DIR.resolve())
print("pandas available:", pd is not None)
print("matplotlib available:", plt is not None)


## 1. Shape inspection — narrate before solving

Before you transform anything, inspect the object type, row count, keys/columns, sample rows, and obvious nulls. This is the fastest way to avoid writing code against the wrong shape.


In [ ]:
def inspect_records(records, name="records", n=3):
    print(f"--- {name} ---")
    print("type:", type(records).__name__)
    try:
        print("len:", len(records))
    except TypeError:
        print("len: n/a")
    for i, row in enumerate(list(records)[:n]):
        print(f"sample[{i}]:", row)
    if records and isinstance(records[0], dict):
        print("keys:", sorted(records[0].keys()))

with open(DATA_DIR / "sample_results.json") as f:
    records = json.load(f)

inspect_records(records, "sample_results")


## 2. JSON loading — validate the top-level shape

Interview mistakes often happen when you assume JSON is a list but it is actually a dictionary, or vice versa. Load, validate, then summarize.


In [ ]:
def load_json_summary(path: str | Path) -> dict:
    with open(path) as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Expected a list of result records")
    models = sorted({row.get("model") for row in data})
    return {"rows": len(data), "models": models, "fields": sorted(data[0].keys())}

summary = load_json_summary(DATA_DIR / "sample_results.json")
pprint(summary)


## 3. CSV loading — stdlib first, pandas when useful

`csv.DictReader` is transparent and interview-friendly. Pandas is excellent once you need fast grouping, joins, charts, or null inspection.


In [ ]:
def read_csv_dicts(path: str | Path) -> list[dict]:
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

rows = read_csv_dicts(DATA_DIR / "telemetry.csv")
inspect_records(rows, "telemetry csv rows")

if pd is not None:
    df = pd.read_csv(DATA_DIR / "telemetry.csv")
    display(df.head()) if "display" in globals() else print(df.head())
    print("\ndtypes:")
    print(df.dtypes)


## 4. JSONL parsing — keep good rows and report bad rows

For JSONL, one malformed line should not always kill the whole task. A practical parser returns parsed records plus a report of failed line numbers.


In [ ]:
def parse_jsonl_with_report(path: str | Path) -> dict:
    parsed = []
    bad_lines = []
    with open(path) as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                parsed.append(json.loads(line))
            except json.JSONDecodeError as exc:
                bad_lines.append({"line_no": line_no, "error": str(exc), "raw": line[:80]})
    return {"records": parsed, "bad_lines": bad_lines, "ok_count": len(parsed), "bad_count": len(bad_lines)}

report = parse_jsonl_with_report(DATA_DIR / "batch_outputs.jsonl")
pprint(report)


## 5. Aggregation — group by model with explicit denominators

Pass rate means successful rows divided by all relevant rows. Say the denominator out loud so the interviewer knows you are not accidentally dividing by successes only.


In [ ]:
def aggregate_by_model(records: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for row in records:
        grouped[row["model"]].append(row)

    output = []
    for model, rows in grouped.items():
        scores = [float(r["score"]) for r in rows]
        passed = [bool(r["passed"]) for r in rows]
        latencies = [float(r["latency_ms"]) for r in rows]
        output.append({
            "model": model,
            "n": len(rows),
            "avg_score": round(sum(scores) / len(scores), 3),
            "pass_rate": round(sum(passed) / len(passed), 3),
            "p50_latency_ms": stats.median(latencies),
            "total_cost_usd": round(sum(float(r["cost_usd"]) for r in rows), 3),
        })
    return sorted(output, key=lambda r: (-r["avg_score"], r["p50_latency_ms"]))

model_summary = aggregate_by_model(records)
pprint(model_summary)


## 6. Pandas aggregation — same logic, more compact

If pandas is available, use named aggregations so the output columns explain themselves. Avoid clever one-liners that hide the denominator.


In [ ]:
if pd is not None:
    results_df = pd.DataFrame(records)
    pandas_summary = (
        results_df.groupby("model")
        .agg(
            n=("task", "count"),
            avg_score=("score", "mean"),
            pass_rate=("passed", "mean"),
            p50_latency_ms=("latency_ms", "median"),
            total_cost_usd=("cost_usd", "sum"),
        )
        .reset_index()
        .sort_values(["avg_score", "p50_latency_ms"], ascending=[False, True])
    )
    display(pandas_summary) if "display" in globals() else print(pandas_summary)
else:
    print("pandas not available; stdlib aggregation above is enough for the interview.")


## 7. Joining metadata — validate unmatched keys

A join is not complete until you check what matched and what did not. In interviews, this is a strong signal that you understand data quality, not just syntax.


In [ ]:
metadata = [
    {"model": "alpha", "provider": "openai", "family": "frontier"},
    {"model": "bravo", "provider": "anthropic", "family": "frontier"},
    {"model": "delta", "provider": "local", "family": "small"},
]

def left_join_models(summary_rows, metadata_rows):
    lookup = {row["model"]: row for row in metadata_rows}
    joined = []
    missing = []
    for row in summary_rows:
        meta = lookup.get(row["model"])
        if meta is None:
            missing.append(row["model"])
            meta = {"provider": None, "family": None}
        joined.append({**row, **meta})
    return joined, missing

joined, missing = left_join_models(model_summary, metadata)
pprint(joined)
print("missing metadata:", missing)


## 8. Visualisation — pick the chart for the question

Visualisation is for understanding, not decoration. Use a histogram for distribution, a line chart for time, a bar chart for grouped comparison, and a scatterplot for trade-offs.


In [ ]:
if pd is not None and plt is not None:
    results_df = pd.DataFrame(records)
    ax = results_df.plot.scatter(x="cost_usd", y="score", c="latency_ms", colormap="viridis", title="Quality vs cost, coloured by latency")
    ax.set_xlabel("Cost per run (USD)")
    ax.set_ylabel("Score")
    plt.show()
else:
    print("If plotting libraries are unavailable, describe the chart you would build:")
    print("Scatterplot: x=cost_usd, y=score, colour=latency_ms, one point per model/task run.")


## 9. Telemetry summary — operational evidence from real runs

Telemetry is observed runtime behaviour. Summarise request count, error rate, p50/p95 latency, cost, and instrumentation gaps such as missing span IDs.


In [ ]:
def percentile(values, p):
    values = sorted(float(v) for v in values)
    if not values:
        return math.nan
    idx = min(len(values) - 1, math.ceil((p / 100) * len(values)) - 1)
    return values[idx]

def summarize_telemetry(rows: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["provider"]].append(row)
    out = []
    for provider, group in grouped.items():
        ok_flags = [200 <= int(r["status_code"]) <= 299 for r in group]
        latencies = [float(r["latency_ms"]) for r in group]
        missing_spans = sum(1 for r in group if not r.get("span_id"))
        out.append({
            "provider": provider,
            "requests": len(group),
            "error_rate": round(1 - (sum(ok_flags) / len(ok_flags)), 3),
            "p50_latency_ms": percentile(latencies, 50),
            "p95_latency_ms": percentile(latencies, 95),
            "total_cost_usd": round(sum(float(r["cost_usd"]) for r in group), 3),
            "missing_spans": missing_spans,
        })
    return sorted(out, key=lambda r: (r["error_rate"], r["p95_latency_ms"]))

telemetry_summary = summarize_telemetry(rows)
pprint(telemetry_summary)


## 10. Benchmark scorecard — compare quality, latency, cost, and sample size

A benchmark recommendation should be constraint-aware. Do not blindly pick the highest score; filter for enough evidence, then compare quality against p95 latency and cost.


In [ ]:
def benchmark_scorecard(records: list[dict], min_n: int = 2) -> list[dict]:
    summary = aggregate_by_model(records)
    for row in summary:
        row["eligible"] = row["n"] >= min_n
        row["recommendation_note"] = (
            "eligible" if row["eligible"] else f"needs at least {min_n} runs before recommendation"
        )
    return sorted(summary, key=lambda r: (not r["eligible"], -r["avg_score"], r["p50_latency_ms"], r["total_cost_usd"]))

scorecard = benchmark_scorecard(records, min_n=2)
pprint(scorecard)
print("Recommendation:", scorecard[0]["model"], "if quality is the primary constraint and sample size is acceptable.")


## 11. Output formatting — make results reviewable

Interviewers like outputs that are deterministic, labelled, and easy to check. Sort rows, round only at presentation time, and include units in labels.


In [ ]:
def markdown_table(rows: list[dict], columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(col, "")) for col in columns) + " |")
    return "\n".join([header, sep] + body)

print(markdown_table(scorecard, ["model", "n", "avg_score", "pass_rate", "p50_latency_ms", "total_cost_usd", "eligible"]))


## 12. Debugging checklist — inspect, isolate, fix, re-run

When a result looks wrong, avoid random edits. Print the smallest relevant slice, verify types, reproduce the bug, then make one fix and rerun the same check.


In [ ]:
buggy_rows = [{"model": "alpha", "score": "9.5"}, {"model": "bravo", "score": "10.0"}]
print("Wrong string sort:", sorted(buggy_rows, key=lambda r: r["score"], reverse=True))

fixed_rows = [{**row, "score": float(row["score"])} for row in buggy_rows]
print("Correct numeric sort:", sorted(fixed_rows, key=lambda r: r["score"], reverse=True))


## 13. Final interview narration template

Use this structure when presenting your answer: input shape, assumptions, transformation plan, sanity checks, output, caveat, and next step. That is what makes the notebook feel like a real working session rather than a pasted solution.


In [ ]:
def narrate_solution(problem, input_shape, assumptions, checks, caveat):
    return f"""
Problem: {problem}
Input shape: {input_shape}
Assumptions: {assumptions}
Checks: {checks}
Caveat: {caveat}
Next step: I would run this on the full file and compare a few rows manually before shipping.
""".strip()

print(narrate_solution(
    problem="Recommend a model from benchmark and telemetry evidence",
    input_shape="5 benchmark rows plus 5 telemetry rows, grouped by model/provider",
    assumptions="score is higher-is-better; p95 latency and cost are constraints, not decorations",
    checks="validated row counts, denominator for pass rate, missing telemetry spans, and numeric score types",
    caveat="sample size is tiny, so this is a method demonstration rather than a statistically strong conclusion",
))
